In [2]:
import arcpy
import os
import datetime

arcpy.env.overwriteOutput = True

gpkg = r"E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg"
out_dir = os.path.dirname(gpkg)

work_gdb = os.path.join(out_dir, "stcube_work.gdb")
out_cube = os.path.join(out_dir, "taxi_line_cube_500m_1h.nc")

CELL_SIZE = 500
TIME_STEP = "1 Hours"

# 北京适用：WGS 1984 UTM Zone 50N，单位米
target_sr = arcpy.SpatialReference(32650)

if not arcpy.Exists(work_gdb):
    arcpy.management.CreateFileGDB(out_dir, "stcube_work.gdb")

arcpy.env.workspace = gpkg

line_layers = []
for fc in arcpy.ListFeatureClasses():
    desc = arcpy.Describe(fc)
    if desc.shapeType == "Polyline":
        line_layers.append(os.path.join(gpkg, fc))

if not line_layers:
    raise RuntimeError("01-07.gpkg 中没有找到线图层。")

merged_lines = os.path.join(work_gdb, "all_trip_lines_wgs84")
arcpy.management.Merge(line_layers, merged_lines)

# start_time 是 Unix 秒时间戳，转成 ArcGIS Date 字段
if "trip_date" not in [f.name for f in arcpy.ListFields(merged_lines)]:
    arcpy.management.AddField(merged_lines, "trip_date", "DATE")

with arcpy.da.UpdateCursor(merged_lines, ["start_time", "trip_date"]) as cursor:
    for start_time, _ in cursor:
        if start_time is None:
            cursor.updateRow([start_time, None])
        else:
            dt = datetime.datetime.fromtimestamp(int(start_time))
            cursor.updateRow([start_time, dt])

# 投影到米制坐标系
projected_lines = os.path.join(work_gdb, "all_trip_lines_utm50n")
arcpy.management.Project(
    in_dataset=merged_lines,
    out_dataset=projected_lines,
    out_coor_system=target_sr
)

desc = arcpy.Describe(projected_lines)
extent = desc.extent

# 创建 500m 鱼网格
fishnet = os.path.join(work_gdb, "fishnet_500m")

origin = f"{extent.XMin} {extent.YMin}"
y_axis = f"{extent.XMin} {extent.YMin + 10}"
corner = f"{extent.XMax} {extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin,
    y_axis_coord=y_axis,
    cell_width=CELL_SIZE,
    cell_height=CELL_SIZE,
    number_rows="",
    number_columns="",
    corner_coord=corner,
    labels="NO_LABELS",
    template=projected_lines,
    geometry_type="POLYGON"
)

if "grid_id" not in [f.name for f in arcpy.ListFields(fishnet)]:
    arcpy.management.AddField(fishnet, "grid_id", "LONG")

oid_field = arcpy.Describe(fishnet).OIDFieldName
arcpy.management.CalculateField(
    fishnet,
    "grid_id",
    f"!{oid_field}!",
    "PYTHON3"
)

# 线与网格相交
intersect_fc = os.path.join(work_gdb, "trip_line_grid_intersect")
arcpy.analysis.PairwiseIntersect(
    in_features=[projected_lines, fishnet],
    out_feature_class=intersect_fc,
    join_attributes="ALL"
)

# 计算每条线在每个网格里的长度，单位米
if "line_m" not in [f.name for f in arcpy.ListFields(intersect_fc)]:
    arcpy.management.AddField(intersect_fc, "line_m", "DOUBLE")

arcpy.management.CalculateGeometryAttributes(
    intersect_fc,
    [["line_m", "LENGTH"]],
    length_unit="METERS"
)

# 创建时空立方体：每个网格每小时出租车轨迹总长度
arcpy.stpm.CreateSpaceTimeCubeDefinedLocations(
    in_features=fishnet,
    output_cube=out_cube,
    location_id="grid_id",
    temporal_aggregation="APPLY_TEMPORAL_AGGREGATION",
    time_field="trip_date",
    time_step_interval=TIME_STEP,
    time_step_alignment="END_TIME",
    reference_time="",
    variables="",
    summary_fields=[["line_m", "SUM", "ZEROS"]],
    in_related_table=intersect_fc,
    related_location_id="grid_id"
)

print("完成时空立方体：", out_cube)

完成时空立方体： E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\taxi_line_cube_500m_1h.nc


In [3]:
arcpy.stpm.MakeSpaceTimeCubeLayer(
    r"E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\taxi_line_cube_500m_1h.nc",
    "taxi_line_cube_500m_1h"
)

<class 'AttributeError'>: module 'arcpy.stpm' has no attribute 'MakeSpaceTimeCubeLayer'

In [4]:
import arcpy
print(arcpy.GetInstallInfo()["Version"])
print(arcpy.ListTools("*SpaceTimeCube*"))
print(arcpy.ListTools("*Cube*"))


3.4.1
['CreateSpaceTimeCube_geoanalytics', 'CreateSpaceTimeCube_stpm', 'CreateSpaceTimeCubeDefinedLocations_stpm', 'CreateSpaceTimeCubeMDRasterLayer_stpm', 'DescribeSpaceTimeCube_stpm', 'SubsetSpaceTimeCube_stpm', 'VisualizeSpaceTimeCube2D_stpm', 'VisualizeSpaceTimeCube3D_stpm']
['CreateSpaceTimeCube_geoanalytics', 'CreateSpaceTimeCube_stpm', 'CreateSpaceTimeCubeDefinedLocations_stpm', 'CreateSpaceTimeCubeMDRasterLayer_stpm', 'DescribeSpaceTimeCube_stpm', 'SubsetSpaceTimeCube_stpm', 'VisualizeSpaceTimeCube2D_stpm', 'VisualizeSpaceTimeCube3D_stpm']


In [5]:
cube = r"E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\taxi_line_cube_500m_1h.nc"

arcpy.DescribeSpaceTimeCube_stpm(cube)
print(arcpy.GetMessages())


开始时间: 2026年7月24日 20:23:46
json:
{"element": "content", "data": ["\u5728 Fri Jul 24 20:19:58 2026 \u4e0a\u4f7f\u7528 ArcGIS Pro 3.4.1 \u521b\u5efa\u7684 \u5df2\u5b9a\u4e49\u4f4d\u7f6e\u7acb\u65b9\u4f53 ", {"element": "sup", "data": "*"}]}
*可以通过“从定义位置创建时空立方体”工具或“通过聚合点创建时空立方体”工具创建定义位置立方体。
json:
[{"element": "accordion", "data": [{"element": "table", "data": [[{"data": "\u8f93\u5165\u8981\u7d20\u65f6\u95f4\u8303\u56f4", "prop": {"rowspan": 2}}, "2017-02-22 11:20:59"], [{"data": "\u5230 2017-03-11 22:36:28", "prop": {"text-align": "right"}}], ["\u5f62\u72b6\u7c7b\u578b", "\u9762"], ["", ""], ["\u65f6\u95f4\u6b65\u957f\u6570", "420"], ["\u65f6\u95f4\u6b65\u957f\u95f4\u9694", "1 \u5c0f\u65f6"], ["\u65f6\u95f4\u6b65\u957f\u5bf9\u9f50", "\u7ec8\u6b62"], ["", ""], ["\u9996\u4e2a\u65f6\u95f4\u6b65\u957f\u65f6\u6001\u504f\u5dee", "74.19%"], [{"data": "\u9996\u4e2a\u65f6\u95f4\u6b65\u957f\u95f4\u9694", "prop": {"rowspan": 4}}, "\u665a\u4e8e"], [{"data": "2017-02-22 10:36:28", "prop": {"text-align":

In [6]:
import arcpy
import os
import datetime

arcpy.env.overwriteOutput = True

# ========== 参数 ==========
gpkg = r"E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg"
out_dir = os.path.dirname(gpkg)

work_gdb = os.path.join(out_dir, "stcube_work_1000m_1d.gdb")
out_cube = os.path.join(out_dir, "taxi_line_cube_1000m_1d.nc")

CELL_SIZE = 1000
TIME_STEP = "1 Days"
REFERENCE_TIME = datetime.datetime(2017, 3, 1, 0, 0, 0)

# 北京：WGS 1984 UTM Zone 50N，单位米
target_sr = arcpy.SpatialReference(32650)

# ========== 创建工作 GDB ==========
if not arcpy.Exists(work_gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(work_gdb))

# ========== 读取 GeoPackage 中所有线图层 ==========
arcpy.env.workspace = gpkg

line_layers = []
for fc in arcpy.ListFeatureClasses():
    desc = arcpy.Describe(fc)
    if desc.shapeType == "Polyline":
        line_layers.append(os.path.join(gpkg, fc))

if not line_layers:
    raise RuntimeError("01-07.gpkg 中没有找到线图层。")

print("找到线图层：")
for lyr in line_layers:
    print(" -", lyr)

# ========== 合并所有线 ==========
merged_lines = os.path.join(work_gdb, "all_trip_lines_wgs84")
arcpy.management.Merge(line_layers, merged_lines)

# ========== Unix 秒时间戳转 Date 字段 ==========
if "trip_date" not in [f.name for f in arcpy.ListFields(merged_lines)]:
    arcpy.management.AddField(merged_lines, "trip_date", "DATE")

with arcpy.da.UpdateCursor(merged_lines, ["start_time", "trip_date"]) as cursor:
    for start_time, _ in cursor:
        if start_time is None:
            cursor.updateRow([start_time, None])
        else:
            dt = datetime.datetime.fromtimestamp(int(start_time))
            cursor.updateRow([start_time, dt])

# ========== 投影到米制坐标系 ==========
projected_lines = os.path.join(work_gdb, "all_trip_lines_utm50n")

arcpy.management.Project(
    in_dataset=merged_lines,
    out_dataset=projected_lines,
    out_coor_system=target_sr
)

# ========== 创建 1 公里鱼网格 ==========
desc = arcpy.Describe(projected_lines)
extent = desc.extent

fishnet = os.path.join(work_gdb, "fishnet_1000m")

origin = f"{extent.XMin} {extent.YMin}"
y_axis = f"{extent.XMin} {extent.YMin + 10}"
corner = f"{extent.XMax} {extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin,
    y_axis_coord=y_axis,
    cell_width=CELL_SIZE,
    cell_height=CELL_SIZE,
    number_rows="",
    number_columns="",
    corner_coord=corner,
    labels="NO_LABELS",
    template=projected_lines,
    geometry_type="POLYGON"
)

# 添加稳定的 grid_id
if "grid_id" not in [f.name for f in arcpy.ListFields(fishnet)]:
    arcpy.management.AddField(fishnet, "grid_id", "LONG")

oid_field = arcpy.Describe(fishnet).OIDFieldName

arcpy.management.CalculateField(
    fishnet,
    "grid_id",
    f"!{oid_field}!",
    "PYTHON3"
)

# ========== 线与网格相交 ==========
intersect_fc = os.path.join(work_gdb, "trip_line_grid_intersect_1000m")

arcpy.analysis.PairwiseIntersect(
    in_features=[projected_lines, fishnet],
    out_feature_class=intersect_fc,
    join_attributes="ALL"
)

# ========== 计算每段线在网格内的长度 ==========
if "line_m" not in [f.name for f in arcpy.ListFields(intersect_fc)]:
    arcpy.management.AddField(intersect_fc, "line_m", "DOUBLE")

arcpy.management.CalculateGeometryAttributes(
    intersect_fc,
    [["line_m", "LENGTH"]],
    length_unit="METERS"
)

# ========== 创建 1 公里 x 1 天时空立方体 ==========
arcpy.CreateSpaceTimeCubeDefinedLocations_stpm(
    fishnet,
    out_cube,
    "grid_id",
    "APPLY_TEMPORAL_AGGREGATION",
    "trip_date",
    TIME_STEP,
    "REFERENCE_TIME",
    REFERENCE_TIME,
    "",
    [["line_m", "SUM", "ZEROS"]],
    intersect_fc,
    "grid_id"
)

print("完成时空立方体：", out_cube)

# ========== 检查立方体信息 ==========
arcpy.DescribeSpaceTimeCube_stpm(out_cube)
print(arcpy.GetMessages())


找到线图层：
 - E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg\main.20170301_trip_lines_filtered
 - E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg\main.20170302_trip_lines_filtered
 - E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg\main.20170303_trip_lines_filtered
 - E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg\main.20170304_trip_lines_filtered
 - E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg\main.20170305_trip_lines_filtered
 - E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg\main.20170306_trip_lines_filtered
 - E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\01-07.gpkg\main.20170307_trip_lines_filtered
完成时空立方体： E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\taxi_line_cube_1000m_1d.nc
开始时间: 2026年7月24日 20:31:21
json:
{"element": "content", "data": ["\u5728 Fri Jul 24 20:31:21 2026 \u4e0a\u4f7f\u7528 ArcGIS Pro 3.4.1 \u521b\u5efa\u7684 \u5df2\u5b9a\u4e49\u4

In [7]:
import arcpy
import os

work_gdb = r"E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\stcube_work_1000m_1d.gdb"
cube = r"E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\taxi_line_cube_1000m_1d.nc"

out_3d = os.path.join(work_gdb, "taxi_line_cube_1000m_1d_3d")

arcpy.VisualizeSpaceTimeCube3D_stpm(
    cube,
    "LINE_M_SUM_ZEROS",
    "VALUE",
    out_3d
)

print("完成 3D 可视化：", out_3d)


完成 3D 可视化： E:\summercamp\出租车数据\1\deliverables\trip\real-trips-line\stcube_work_1000m_1d.gdb\taxi_line_cube_1000m_1d_3d
